# 12 · Speed-conditioned healthy thresholds (Stage R1)

## Purpose
Stage R0 found that healthy false alarms concentrate at speeds outside the calibration normals' speed range. R1 tests that
hypothesis with a score transformation, separately for **AE-6** and **B1s**: s_std = (s − m(speed)) / d(speed), with m and
d fitted against log speed on each run's healthy **fit** normals only, and the threshold on s_std set from the **same
calibration normals** by the existing maximum rule (R0's tie rule carried forward). Evaluation uses the Stage 3C seed-42
LOCO runs and their saved assignment (CLAUDE.md §7, Stage R1, declared before running). Exploratory development data
only: no training, no new model, no deployed threshold or app change, no reserved recording.

The fit normals had no saved scores, so they were re-scored once (`scripts/32_r1_fit_normal_scores.py`, Option A:
fit normals only; AE-6 inference with the saved 3C weights; B1s the deterministic 3D refit, used only after it reproduced
every run's saved cal/eval scores within 1e-12).

**In-sample bias (applies to every standardised number below).** m(speed) and d(speed) are fitted on each run's healthy FIT normals, the recordings AE-6 was trained on and B1s was fitted on, so their scores are in-sample and optimistically low. That biases the standardised score upward for calibration and evaluation recordings, which works against the new method. A positive result would therefore be conservative and meaningful; a negative result is partly attributable to this bias and does not by itself refute speed-conditioning. The clean alternative (out-of-fold scores on the fit normals) does not exist in this project.

**Speed caveat.** 25 healthy recordings from one folder (one recording = 4 percentage points); speed and recording identity are completely confounded; only 5 recordings lie outside the bundle's calibration-normal speed range. R0's hypothesis remains a hypothesis.

## Inputs
`metadata.csv` (development pool, measured speed `fr_measured_hz`); `results/q1q3/q3_fit_cal_eval_assignment.csv`;
`results/scores/cnn6ae6_s42_loco_decisions.parquet` (top class, MSP, Mahalanobis, AE-6);
`results/scores/b1s_nomic_loco_q3fit_scores.parquet` (B1s, τ_b); `results/loco/decision_thresholds.csv` (τ_p, τ_d, τ_e);
`results/research/r1_fit_normal_scores.parquet` and `r1_reproduction.json` (the re-score); `artifacts/bundle/manifest.json`
(read only: the bundle's calibration-normal speed range, a secondary speed band). Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration and metadata, prints the config and split hashes, and checks the healthy counts against CLAUDE.md §3 (49 normals, 25 in the development pool, one folder); it stops with an error if a count differs. It also reads the bundle's calibration-normal speed range (read only), used as a secondary speed band.</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from ml.config import load_config, config_sha256
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import development_pool, folder_proxy, split_hash
from ml.arbitrator import decide
from ml.rejection import snap_ties, rate, speed_band, TIE_EPS
from ml.q1 import mild_mask
from ml.speed_calibration import standardise_run, exceeds, matched_threshold

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns)
normals = pool[pool.class10 == "normal"]
assert (meta.class10 == "normal").sum() == 49 and len(normals) == 25 and len(set(folder_proxy(normals))) == 1, \
    "normal counts differ from CLAUDE.md §3: stop and ask"
manifest = json.loads((ROOT / "artifacts/bundle/manifest.json").read_text())
B_LO, B_HI = manifest["data"]["calibration_normal_speed_range_hz"]
print(f"25 development normals, one folder; speeds {normals.fr_measured_hz.min():.2f}-{normals.fr_measured_hz.max():.2f} Hz; "
      f"bundle calibration-normal range {B_LO:.2f}-{B_HI:.2f} Hz; outside it: "
      f"{int(((normals.fr_measured_hz < B_LO) | (normals.fr_measured_hz > B_HI)).sum())} recordings")

## The re-score reproduced the saved scores
Before any fit-normal score is used, the re-score recomputed every run's saved calibration and evaluation scores. B1s must
match within 1e-12 (R0's tie tolerance); AE-6 within 1e-5 × τ_e (declared; MPS is not bit-deterministic). A run that failed
would be infeasible for that detector, with no approximation substituted.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads the reproduction record written by the re-score script and prints, per detector, how many of the 27 runs reproduced their saved scores and the largest absolute difference found. Any run listed as infeasible is excluded for that detector below.</div>

In [ ]:
rep = json.loads((R / "research/r1_reproduction.json").read_text())
for det in ("ae", "b1s"):
    ev = rep[det]
    print(f"{det}: {sum(e['reproduced'] for e in ev.values())}/27 runs reproduced; saved rows checked "
          f"{sum(e['n_saved'] for e in ev.values())}; max |diff| {max(e['max_abs_diff'] for e in ev.values()):.2e}; "
          f"infeasible: {rep['infeasible'][det] or 'none'}")
INFEASIBLE = {det: set(rep["infeasible"][det]) for det in ("ae", "b1s")}

## One table: saved scores plus the fit normals
Each of the 27 runs contributes its fit normals (re-scored, in-sample), its calibration recordings and its evaluation
recordings (seen-folder `eval`, unseen-fold `foldk_known`, withheld `foldk_withheld`). Normals recur across runs.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> joins the saved Stage 3C scores (top class, MSP, Mahalanobis, AE-6), the saved B1s scores and the run thresholds, adds the re-scored fit normals and each recording's measured speed, and counts rows by role. The evaluation normals must be the same 299 instances of 25 recordings used in R0.</div>

In [ ]:
key = ["withheld", "outer_fold", "rec_id"]
dec = pd.read_parquet(R / "scores/cnn6ae6_s42_loco_decisions.parquet")[
    key + ["pred", "msp", "maha", "ae", "role", "class10", "severity"]]
b1s = pd.read_parquet(R / "scores/b1s_nomic_loco_q3fit_scores.parquet")[key + ["b1s", "tau_b"]]
thr = pd.read_csv(R / "loco/decision_thresholds.csv")[["withheld", "outer_fold", "tau_p", "tau_d", "tau_e"]]
saved = dec.merge(b1s, on=key, validate="1:1").merge(thr, on=["withheld", "outer_fold"])
assert len(saved) == len(dec) == len(b1s)
fit = pd.read_parquet(R / "research/r1_fit_normal_scores.parquet").assign(class10="normal")
df = pd.concat([saved, fit[key + ["ae", "b1s", "role", "class10"]]], ignore_index=True)
info = meta.set_index("rec_id")
df["speed_hz"] = info.fr_measured_hz.loc[df.rec_id].to_numpy()
df["fault_family"] = info.fault_family.loc[df.rec_id].to_numpy()
EVAL = ("eval", "foldk_known", "foldk_withheld")
df["is_eval"] = df.role.isin(EVAL)
hn = df[df.is_eval & (df.class10 == "normal")]
assert len(hn) == 299 and hn.rec_id.nunique() == 25
print(df.assign(healthy=df.class10 == "normal").groupby(["role", "healthy"]).size().unstack(fill_value=0))
print("fit normals per run:", fit.groupby(["withheld", "outer_fold"]).size().describe()[["min", "50%", "max"]].to_dict())

## How large is the in-sample bias?
The fit normals' raw scores are compared with the calibration and evaluation normals' raw scores of the same runs. A
lower level on the fit normals is the optimism that the standardisation inherits.

**In-sample bias (applies to every standardised number below).** m(speed) and d(speed) are fitted on each run's healthy FIT normals, the recordings AE-6 was trained on and B1s was fitted on, so their scores are in-sample and optimistically low. That biases the standardised score upward for calibration and evaluation recordings, which works against the new method. A positive result would therefore be conservative and meaningful; a negative result is partly attributable to this bias and does not by itself refute speed-conditioning. The clean alternative (out-of-fold scores on the fit normals) does not exist in this project.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> compares, per detector, the median raw score of the healthy fit normals (in-sample) with that of the calibration and evaluation normals (not used to fit the detector), pooled over the 27 runs. A ratio below 1 means the fit normals look healthier than new healthy recordings, which is the in-sample optimism.</div>

In [ ]:
nrm = df[df.class10 == "normal"].assign(group=lambda x: np.where(x.role == "fit", "fit (in-sample)",
                                                                  np.where(x.role == "cal", "calibration", "evaluation")))
bias = nrm.groupby("group")[["ae", "b1s"]].median()
print(bias.round(4))
BIAS_RATIO = {d: float(bias.loc["fit (in-sample)", d] / bias.loc["evaluation", d]) for d in ("ae", "b1s")}
print("median fit / median evaluation:", {k: round(v, 3) for k, v in BIAS_RATIO.items()})

## Standardise per run and per detector
For every run and each detector: m (degree 2, the cap, and degree 1) and d (degree 1, log link) are fitted on that run's
fit normals only (`ml.speed_calibration.fit_for_run`), every row is standardised, and τ_std is the maximum over the run's
calibration normals. Speeds are clipped to the fit normals' range before m and d are evaluated.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> standardises the AE-6 and B1s scores of every run with the two declared variants (m of degree 2 or 1, d of degree 1) and records each run's threshold on the standardised score and its fitted coefficients. The raw score keeps its saved threshold (τ_e or τ_b). Infeasible runs, if any, are skipped for that detector.</div>

In [ ]:
TAU = {"ae": "tau_e", "b1s": "tau_b"}
VARIANTS = {"m2d1": 2, "m1d1": 1}
models = []
parts = []
for (c, k), g in df.groupby(["withheld", "outer_fold"]):
    g = g.copy()
    for det in ("ae", "b1s"):
        if f"{c}/{k}" in INFEASIBLE[det]:
            continue
        g[f"{det}__raw"] = g[det]
        g[f"{det}__raw__thr"] = float(g[TAU[det]].dropna().iloc[0])
        for var, deg in VARIANTS.items():
            s, tau, m = standardise_run(g, det, deg)
            g[f"{det}__{var}"] = s
            g[f"{det}__{var}__thr"] = tau
            models.append({"withheld": c, "outer_fold": k, "detector": det, "variant": var, "tau_std": tau,
                           "m_coef": list(m.m_coef), "d_coef": list(m.d_coef), "fit_speed_hz": [float(np.exp(m.x_lo)), float(np.exp(m.x_hi))],
                           "n_fit": m.n_fit})
    parts.append(g)
df = pd.concat(parts, ignore_index=True)
cal_rng = df[(df.role == "cal") & (df.class10 == "normal")].groupby(["withheld", "outer_fold"]).speed_hz.agg(["min", "max"])
df = df.join(cal_rng.rename(columns={"min": "cal_lo", "max": "cal_hi"}), on=["withheld", "outer_fold"])
models = pd.DataFrame(models)
print(models.groupby(["detector", "variant"]).tau_std.describe()[["min", "50%", "max"]].round(3))

## Figure: healthy scores against speed, raw and standardised
**Speed caveat.** 25 healthy recordings from one folder (one recording = 4 percentage points); speed and recording identity are completely confounded; only 5 recordings lie outside the bundle's calibration-normal speed range. R0's hypothesis remains a hypothesis.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> plots, for each detector, the healthy scores of every run against measured speed: raw on the left (divided by the run's threshold, so 1 is the limit) and standardised (degree-1 variant, divided by its run threshold τ_std, which is positive in every run) on the right. Grey dots are the fit normals (in-sample), blue the calibration normals, orange the evaluation normals; the shaded band is the bundle's calibration-normal speed range. Points above the dashed line at 1 are false alarms. The figure is saved for the report.</div>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
style = {"fit": ("#9e9e9e", "fit normals (in-sample)"), "cal": ("#1f77b4", "calibration normals"),
         "eval": ("#ff7f0e", "evaluation normals")}
for i, det in enumerate(("ae", "b1s")):
    for j, var in enumerate(("raw", "m1d1")):
        ax = axes[i, j]
        ax.axvspan(B_LO, B_HI, color="#e8f5e9", zorder=0, label="bundle calibration-normal speeds")
        for grp, (col, lab) in style.items():
            sel = (df.class10 == "normal") & ((df.role == grp) if grp != "eval" else df.is_eval)
            assert (df.loc[sel, f"{det}__{var}__thr"] > 0).all()
            y = df.loc[sel, f"{det}__{var}"] / df.loc[sel, f"{det}__{var}__thr"]
            ax.scatter(df.loc[sel, "speed_hz"], y, s=12, alpha=0.55, color=col, label=lab)
        ax.axhline(1, ls="--", color="k", lw=1)
        ax.set_title(f"{'AE-6' if det == 'ae' else 'B1s'} · {'raw score / τ' if var == 'raw' else 'standardised (m1d1): s_std / τ_std'}")
        ax.set_ylabel("score relative to the run threshold (1 = limit)")
        if i == 1:
            ax.set_xlabel("measured shaft speed (Hz)")
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("Healthy scores vs speed, 27 LOCO runs (25 recordings, one folder; speed and recording are confounded)", fontsize=10)
fig.tight_layout()
(R / "figures").mkdir(exist_ok=True)
fig.savefig(R / "figures/r1_score_vs_speed.png", dpi=150)
plt.show()

## Per-run evaluation
For each run, detector and variant (raw, m2d1, m1d1), on the run's evaluation recordings: AUROC healthy vs faulty
(threshold-free, the primary measure); healthy false alarms at the calibrated threshold; fault detection at the calibrated
threshold; and fault detection at a **matched false-alarm count**: the std threshold is placed so that it flags exactly as
many evaluation normals as the raw rule flags at its calibrated τ (a comparison point only, never a deployable threshold).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loops over the 27 runs and, for each detector and variant, computes the AUROC (faults vs healthy evaluation recordings), the number of healthy evaluation instances above the threshold, and the share of fault instances above the calibrated threshold and above the matched threshold. It also stores, per evaluation instance, whether it was flagged, for the breakdowns below.</div>

In [ ]:
VARS = ("raw", "m2d1", "m1d1")
per_run, flags = [], []
for (c, k), g in df[df.is_eval].groupby(["withheld", "outer_fold"]):
    y = (g.class10 != "normal").to_numpy()
    for det in ("ae", "b1s"):
        if f"{det}__raw" not in g or g[f"{det}__raw"].isna().any():
            continue
        fa_raw = int(exceeds(g.loc[~y, f"{det}__raw"], g[f"{det}__raw__thr"].iloc[0]).sum())
        for var in VARS:
            s, t = g[f"{det}__{var}"].to_numpy(), float(g[f"{det}__{var}__thr"].iloc[0])
            mt = matched_threshold(s[~y], fa_raw)
            cal_flag, m_flag = exceeds(s, t), exceeds(s, mt)
            per_run.append({"withheld": c, "outer_fold": k, "detector": det, "variant": var, "auroc": roc_auc_score(y, s),
                            "threshold": t, "healthy_instances": int((~y).sum()), "healthy_false_alarms": int(cal_flag[~y].sum()),
                            "fault_instances": int(y.sum()), "detected_calibrated": int(cal_flag[y].sum()),
                            "matched_false_alarms": int(m_flag[~y].sum()), "detected_matched": int(m_flag[y].sum())})
            flags.append(pd.DataFrame({"withheld": c, "outer_fold": k, "rec_id": g.rec_id.to_numpy(), "detector": det,
                                       "variant": var, "flag_calibrated": cal_flag, "flag_matched": m_flag}))
per_run = pd.DataFrame(per_run)
flags = pd.concat(flags, ignore_index=True)
assert (per_run.matched_false_alarms == per_run.groupby(["withheld", "outer_fold", "detector"]).healthy_false_alarms
        .transform("first")).all(), "matched threshold does not match the raw false-alarm count"
print(per_run.groupby(["detector", "variant"]).size())

## (i) Primary: per-fold AUROC, raw vs standardised
Per outer fold, the mean AUROC over its 9 runs; then the mean and spread (min–max) across the 3 folds. Pre-declared
degree rule: if m2 beats m1 only marginally (macro gain < 0.01, or higher in fewer than 2 of 3 fold means), degree 1 is
preferred.

**In-sample bias (applies to every standardised number below).** m(speed) and d(speed) are fitted on each run's healthy FIT normals, the recordings AE-6 was trained on and B1s was fitted on, so their scores are in-sample and optimistically low. That biases the standardised score upward for calibration and evaluation recordings, which works against the new method. A positive result would therefore be conservative and meaningful; a negative result is partly attributable to this bias and does not by itself refute speed-conditioning. The clean alternative (out-of-fold scores on the fit normals) does not exist in this project.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> averages the per-run AUROC within each outer fold, then reports the mean and the range across the three folds for each detector and variant, the difference from raw in each fold, and applies the declared degree-preference rule.</div>

In [ ]:
fold = per_run.groupby(["detector", "variant", "outer_fold"]).auroc.mean().unstack()
summ = pd.DataFrame({"fold_mean": fold.mean(axis=1), "fold_min": fold.min(axis=1), "fold_max": fold.max(axis=1),
                     "runs_mean": per_run.groupby(["detector", "variant"]).auroc.mean()})
print(pd.concat([fold, summ], axis=1).round(4))
DIFF = {}
for det in ("ae", "b1s"):
    for var in ("m2d1", "m1d1"):
        dlt = fold.loc[(det, var)] - fold.loc[(det, "raw")]
        DIFF[(det, var)] = dlt
        print(f"{det} {var} − raw per fold: {dlt.round(4).to_dict()}; runs where std > raw: "
              f"{int((per_run[(per_run.detector == det) & (per_run.variant == var)].auroc.to_numpy() > per_run[(per_run.detector == det) & (per_run.variant == 'raw')].auroc.to_numpy()).sum())}/27")
PREFERRED = {}
for det in ("ae", "b1s"):
    gain = fold.loc[(det, "m2d1")] - fold.loc[(det, "m1d1")]
    PREFERRED[det] = "m2d1" if (gain.mean() >= 0.01 and (gain > 0).sum() >= 2) else "m1d1"
    print(f"{det}: m2 − m1 macro gain {gain.mean():+.4f}, higher in {(gain > 0).sum()}/3 folds → preferred {PREFERRED[det]}")

## (ii) Healthy false alarms by speed band
The same 299 evaluation instances of 25 recordings as R0; a recording counts once if any of its instances is flagged.
Bands: below / inside / above the run's own calibration normals' speed range (primary; those normals set the threshold)
and the bundle's calibration-normal range (secondary).

**Speed caveat.** 25 healthy recordings from one folder (one recording = 4 percentage points); speed and recording identity are completely confounded; only 5 recordings lie outside the bundle's calibration-normal speed range. R0's hypothesis remains a hypothesis.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> for each detector and variant, counts the healthy evaluation instances above the calibrated threshold, by speed band (relative to each run's calibration-normal speeds, and to the bundle's range), as instances and as distinct recordings with 95 % Clopper–Pearson intervals. It also builds the per-instance healthy table written to r1_healthy_outcomes.csv.</div>

In [ ]:
h = df[df.is_eval & (df.class10 == "normal")][["withheld", "outer_fold", "rec_id", "role", "speed_hz", "cal_lo", "cal_hi",
                                                 "pred", "msp", "maha", "tau_p", "tau_d"] +
                                                [f"{d}__{v}{s}" for d in ("ae", "b1s") for v in VARS for s in ("", "__thr")]]
h["band_run"] = np.where(h.speed_hz < h.cal_lo, "below", np.where(h.speed_hz > h.cal_hi, "above", "inside"))
h["band_bundle"] = speed_band(h.speed_hz, B_LO, B_HI)
hf = h.merge(flags, on=["withheld", "outer_fold", "rec_id"])
rows = []
for band_col in ("band_run", "band_bundle"):
    for (det, var, band), g in hf.groupby(["detector", "variant", band_col]):
        rows.append({"bands": band_col, "detector": det, "variant": var, "band": band,
                     "instances": rate(int(g.flag_calibrated.sum()), len(g)),
                     "recordings": rate(int(g.groupby("rec_id").flag_calibrated.any().sum()), g.rec_id.nunique())})
    for (det, var), g in hf.groupby(["detector", "variant"]):
        rows.append({"bands": band_col, "detector": det, "variant": var, "band": "all",
                     "instances": rate(int(g.flag_calibrated.sum()), len(g)),
                     "recordings": rate(int(g.groupby("rec_id").flag_calibrated.any().sum()), g.rec_id.nunique())})
FA = pd.DataFrame(rows)
show = FA.assign(inst=FA.instances.map(lambda r: f"{r['k']}/{r['n']}"), rec=FA.recordings.map(lambda r: f"{r['k']}/{r['n']} [{r['ci95'][0]:.2f}, {r['ci95'][1]:.2f}]"))
for b in ("band_run", "band_bundle"):
    print(b); print(show[show.bands == b].pivot_table(index=["detector", "band"], columns="variant", values=["inst", "rec"], aggfunc="first").to_string())

## (iii) Cost side: fault detection at a matched healthy false-alarm count
The mandatory check that a lower false-alarm count is not bought with missed faults: at the same number of flagged
evaluation normals per run as the raw rule, how many fault instances does each score flag? By fault class and at the
mildest settings (`anomaly.mildest`: imbalance 6 g, misalignments 0.5 / 0.51 mm, bearings 0 g added mass). Also shown:
detection at the calibrated thresholds.

**In-sample bias (applies to every standardised number below).** m(speed) and d(speed) are fitted on each run's healthy FIT normals, the recordings AE-6 was trained on and B1s was fitted on, so their scores are in-sample and optimistically low. That biases the standardised score upward for calibration and evaluation recordings, which works against the new method. A positive result would therefore be conservative and meaningful; a negative result is partly attributable to this bias and does not by itself refute speed-conditioning. The clean alternative (out-of-fold scores on the fit normals) does not exist in this project.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> joins the fault flags to the fault evaluation instances and reports, per detector and variant, the share of fault instances detected at the matched false-alarm count and at the calibrated threshold: pooled, per fault class, and at the mildest setting of each family (instance counts, with the number of distinct recordings).</div>

In [ ]:
fe = df[df.is_eval & (df.class10 != "normal")][["withheld", "outer_fold", "rec_id", "class10", "fault_family", "severity"]]
ff = fe.merge(flags, on=["withheld", "outer_fold", "rec_id"])
mild = mild_mask(fe.drop_duplicates("rec_id").set_index("rec_id"), cfg.anomaly)
ff["mild"] = ""
for lab, m in mild.items():
    ff.loc[ff.rec_id.isin(m[m].index), "mild"] = lab
def det_table(by):
    t = ff.groupby(["detector", "variant"] + by).agg(n=("flag_matched", "size"), recs=("rec_id", "nunique"),
                                                      matched=("flag_matched", "mean"), calibrated=("flag_calibrated", "mean"))
    return t
pooled = det_table([])
print("pooled fault detection:"); print(pooled.round(4))
by_class = det_table(["class10"])["matched"].unstack(["detector", "variant"])
print("\nmatched false-alarm count, by fault class:"); print(by_class.round(3))
by_mild = ff[ff.mild != ""].groupby(["detector", "variant", "mild"]).agg(n=("flag_matched", "size"), recs=("rec_id", "nunique"),
                                                                        matched=("flag_matched", "mean"))["matched"].unstack(["detector", "variant"])
print("\nmildest settings, matched:"); print(by_mild.round(3))
print("mildest-setting instances / recordings:", ff[(ff.mild != "") & (ff.detector == "ae") & (ff.variant == "raw")].groupby("mild").agg(
    n=("rec_id", "size"), recs=("rec_id", "nunique")).to_dict("index"))

## (iv) Three-way outcomes, old vs new
Rules A, B, C, D and B_with_caution (`ml.arbitrator.decide`) on the evaluation recordings with the saved top class, MSP and
Mahalanobis and their thresholds unchanged; AE-6 and B1s either raw with τ_e, τ_b (old) or standardised with τ_std (new;
both detectors take the same variant). R0's tie rule is applied before deciding. Rule B's decisions do not use AE-6 or
B1s, so under B and B_with_caution only the caution changes.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> recomputes the decision of every evaluation instance under each rule, first with the raw AE-6 and B1s scores and thresholds, then with the standardised ones, and counts healthy / healthy with caution / known fault / unknown, separately for healthy and fault recordings.</div>

In [ ]:
def outcomes(frame, var):
    x = frame.assign(ae=frame[f"ae__{var}"], b1s=frame[f"b1s__{var}"], tau_e=frame[f"ae__{var}__thr"], tau_b=frame[f"b1s__{var}__thr"])
    x = snap_ties(x)
    res = {}
    for rule in ("A", "B", "C", "D", "B_with_caution"):
        out = []
        for r in x.itertuples():
            d = decide(rule, r.pred, float(r.msp), float(r.maha), float(r.ae), float(r.b1s),
                       {"tau_p": r.tau_p, "tau_d": r.tau_d, "tau_e": r.tau_e, "tau_b": r.tau_b})
            out.append("healthy_with_caution" if d["caution"] else d["decision"])
        res[rule] = out
    return pd.DataFrame(res, index=frame.index)
ev = df[df.is_eval].copy()
OUT3 = {var: outcomes(ev, var) for var in VARS}
tw = []
for var, o in OUT3.items():
    for rule in o.columns:
        for grp, sel in (("healthy", ev.class10 == "normal"), ("fault", ev.class10 != "normal")):
            vc = o.loc[sel, rule].value_counts()
            tw.append({"variant": var, "rule": rule, "recordings": grp, **{k: int(vc.get(k, 0)) for k in
                       ("healthy", "healthy_with_caution", "known_fault", "unknown")}})
TW = pd.DataFrame(tw)
print(TW.pivot_table(index=["recordings", "rule"], columns="variant", values=["healthy", "healthy_with_caution", "unknown"]).to_string())

## Pre-declared reading
Per detector and variant: **helps** only if (a) the std AUROC fold mean is ≥ raw in all 3 folds and higher on average,
(b) healthy false alarms (distinct recordings) at the calibrated threshold do not rise, and (c) pooled fault detection at
the matched false-alarm count is not lower; **does not help** if the std AUROC fold mean is lower in all 3 folds;
otherwise **too thin to tell**.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the three declared criteria to each detector and variant and prints the reading, with the numbers each criterion uses.</div>

In [ ]:
READING = {}
fa_all = FA[(FA.bands == "band_run") & (FA.band == "all")].set_index(["detector", "variant"])
for det in ("ae", "b1s"):
    for var in ("m2d1", "m1d1"):
        dlt = DIFF[(det, var)]
        a = bool((dlt >= 0).all() and dlt.mean() > 0)
        b = fa_all.loc[(det, var), "recordings"]["k"] <= fa_all.loc[(det, "raw"), "recordings"]["k"]
        cc = pooled.loc[(det, var), "matched"] >= pooled.loc[(det, "raw"), "matched"]
        verdict = "helps" if (a and b and cc) else ("does not help" if (dlt < 0).all() else "too thin to tell")
        READING[(det, var)] = {"auroc_fold_diff": dlt.round(5).to_dict(), "a_auroc": a, "b_fa_not_up": bool(b),
                               "c_detection_not_lower": bool(cc), "reading": verdict}
        print(f"{det:4s} {var}: {verdict:17s} | AUROC Δ per fold {dlt.round(4).to_dict()} | FA recordings "
              f"{fa_all.loc[(det, var), 'recordings']['k']} vs raw {fa_all.loc[(det, 'raw'), 'recordings']['k']} | matched detection "
              f"{pooled.loc[(det, var), 'matched']:.4f} vs raw {pooled.loc[(det, 'raw'), 'matched']:.4f}")

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the outputs: the per-run table (r1_per_fold.csv), one row per healthy evaluation instance with its raw and standardised scores, thresholds, flags, speed bands and old / new outcomes (r1_healthy_outcomes.csv), and a JSON summary with the reproduction evidence, the in-sample-bias statement, the caveats, per-fold AUROC, false alarms by band, detection at the matched count, three-way counts, per-run thresholds and coefficients, and the reading.</div>

In [ ]:
out = R / "research"
per_run.to_csv(out / "r1_per_fold.csv", index=False)
hw = h.copy()
for var, o in OUT3.items():
    hw = hw.join(o.add_prefix(f"outcome_{var}__"), how="left")
fl = flags.pivot_table(index=["withheld", "outer_fold", "rec_id"], columns=["detector", "variant"], values="flag_calibrated", aggfunc="first")
fl.columns = [f"flag_{d}__{v}" for d, v in fl.columns]
hw = hw.merge(fl.reset_index(), on=["withheld", "outer_fold", "rec_id"])
hw.to_csv(out / "r1_healthy_outcomes.csv", index=False)
BIAS_TEXT = ("In-sample bias: m(speed) and d(speed) are fitted on the healthy fit normals, which AE-6 was trained on and B1s "
             "was fitted on, so their scores are in-sample and optimistically low; this biases the standardised score upward "
             "for calibration and evaluation recordings, against the new method. A positive result is therefore conservative "
             "and meaningful; a negative result is partly attributable to this bias and does not by itself refute "
             "speed-conditioning; the clean alternative (out-of-fold scores on the fit normals) does not exist in this project.")
CAVEAT_TEXT = ("25 healthy recordings from one folder (one recording = 4 percentage points); speed and recording identity are "
               "completely confounded; only 5 recordings lie outside the bundle calibration-normal speed range; R0's "
               "hypothesis remains a hypothesis.")
summary = {
    "stage": "R1", "what": "speed-conditioned healthy thresholds, AE-6 and B1s separately (exploratory, development data, "
    "Stage 3C seed-42 LOCO runs)", "config_sha256": config_sha256(cfg), "split_hash": split_hash(meta),
    "in_sample_bias": BIAS_TEXT, "caveat": CAVEAT_TEXT, "median_fit_over_eval_normal_score": BIAS_RATIO,
    "reproduction": rep, "tie_eps": TIE_EPS, "preferred_variant": PREFERRED,
    "primary_auroc_per_fold": {f"{d}/{v}": {"per_fold": fold.loc[(d, v)].round(6).to_dict(), "fold_mean": float(summ.loc[(d, v), "fold_mean"]),
                                          "fold_range": [float(summ.loc[(d, v), "fold_min"]), float(summ.loc[(d, v), "fold_max"])],
                                          "note": BIAS_TEXT if v != "raw" else "saved scores"} for d, v in fold.index},
    "healthy_false_alarms": FA.to_dict("records"),
    "fault_detection_pooled": {f"{d}/{v}": {"n_instances": int(r.n), "n_recordings": int(r.recs), "matched": float(r.matched),
                                            "calibrated": float(r.calibrated)} for (d, v), r in pooled.iterrows()},
    "fault_detection_matched_by_class": {f"{d}/{v}": by_class[(d, v)].round(6).to_dict() for d, v in by_class.columns},
    "fault_detection_matched_mildest": {f"{d}/{v}": by_mild[(d, v)].round(6).to_dict() for d, v in by_mild.columns},
    "three_way": TW.to_dict("records"),
    "per_run_models": models.to_dict("records"),
    "reading": {f"{d}/{v}": {**r, "note": BIAS_TEXT} for (d, v), r in READING.items()},
    "infeasible_runs": rep["infeasible"],
}
(out / "r1_speed_thresholds.json").write_text(json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("written:", *[p.name for p in (out / "r1_per_fold.csv", out / "r1_healthy_outcomes.csv", out / "r1_speed_thresholds.json")],
      "and figures/r1_score_vs_speed.png")

## Findings

Exploratory development results, Stage 3C seed-42 LOCO runs. **Every standardised number carries the in-sample bias:** m and
d were fitted on the fit normals, which AE-6 was trained on and B1s was fitted on, so their scores are optimistically low.
That bias works against the standardised score, so a negative result is partly attributable to it and does not by itself
refute speed-conditioning; the clean alternative (out-of-fold fit-normal scores) does not exist in this project.
**Speed caveat:** 25 healthy recordings from one folder (one recording = 4 points); speed and recording identity are
completely confounded; only 5 recordings lie outside the bundle's calibration-normal speed range.

- **The re-score is exact, and the bias is large.** All 27 runs reproduced their saved cal/eval scores (11,548 rows; max
  |diff| 0 for AE-6 and for B1s), so no run is infeasible; 305 fit-normal scores (10–12 per run) were added. Median
  fit-normal score / median evaluation-normal score: **AE-6 0.49, B1s 0.85**. For AE-6 the fitted scale d is therefore far
  too small for new healthy recordings (τ_std median ≈ 47 d-units).
- **Primary, per-fold AUROC healthy vs faulty (in-sample-biased against the standardised score):** AE-6 raw fold mean
  **0.987** (range 0.984–0.990) vs **0.968** (0.959–0.974) m1d1 and 0.971 m2d1: lower in all 3 folds (Δ −0.010 to −0.026;
  higher in only 5–6 of 27 runs). B1s raw **0.990** (0.983–0.996) vs **0.986** (0.976–0.995) m1d1 and 0.980 m2d1: lower in
  all 3 folds (Δ −0.001 to −0.020; m1d1 higher in 1 of 27 runs). Degree 1 is preferred for both detectors (AE-6 m2 − m1
  +0.0035 < 0.01; B1s m2 is worse). **Pre-declared reading: "does not help" for both detectors and both variants**, with
  the bias caveat above.
- **Healthy false alarms (299 instances / 25 recordings), calibrated thresholds:** AE-6 73 → 68 instances (m1d1), but
  distinct recordings **16 → 17 of 25**; the change is a shift, not a reduction: below the runs' calibration speed range
  43 → 31 of 60 instances, above it 13 → 20 of 73, inside 17 → 17 of 166; all 7 below-range recordings stay flagged.
  B1s 69 → 68 instances, **14 → 14** recordings. With 25 recordings from one folder, none of these differences is
  distinguishable from recording identity.
- **Cost side, fault detection at a matched healthy false-alarm count (mandatory):** AE-6 pooled **0.984 → 0.938** (m1d1;
  9,098 fault instances, 779 recordings); horizontal misalignment 0.844 → 0.775, and at the mildest settings horizontal
  misalignment 0.5 mm 0.663 → 0.601, overhang outer 0 g 0.997 → 0.834, underhang cage 0 g 0.993 → 0.878. B1s pooled
  **0.987 → 0.981**; horizontal misalignment 0.5 mm 0.914 → 0.897. The standardised scores cost fault detection without a
  false-alarm gain.
- **Three-way outcomes:** rule B ignores AE-6 and B1s, so under B_with_caution only cautions change on healthy
  recordings: plain healthy 198 → 190, caution 52 → 60, unknown 49 → 49 (m1d1). Rules that use the detectors get worse on
  faults: fault instances called healthy under D 2 → 4, under A 29 → 59.
- **What this does and does not show.** On these data, conditioning AE-6 and B1s on speed with a model fitted on the fit
  normals does not improve either detector and costs detection; B1s is already a speed-polynomial detector, so it had
  little speed trend left to remove. R0's hypothesis remains a hypothesis: this test is **negative but weak**, because the
  in-sample fit normals understate healthy spread and speed cannot be separated from recording identity in 25 recordings
  from one folder. A fair test needs out-of-fold healthy scores or more healthy recordings across speeds; neither exists
  in this project. Nothing is deployed or changed.


## Outputs written
- `results/research/r1_speed_thresholds.json`: summary (reproduction evidence, in-sample bias, caveats, per-fold AUROC,
  false alarms by speed band, detection at matched false-alarm count, three-way counts, per-run τ_std and coefficients,
  the reading).
- `results/research/r1_per_fold.csv`: one row per run × detector × variant.
- `results/research/r1_healthy_outcomes.csv`: one row per healthy evaluation instance (299).
- `results/figures/r1_score_vs_speed.png`.
- Written earlier by `scripts/32_r1_fit_normal_scores.py`: `results/research/r1_fit_normal_scores.parquet`,
  `results/research/r1_reproduction.json`.